# FraudLens Data Quality Checks

Use this notebook after regeneration or benchmark processing to answer a few focused questions:

- Do Raw, Bronze, Silver, and Gold row counts match?
- Do labels align with the Gold features we expect?
- Are the worst outliers understandable for the current dataset mode?


In [ ]:
from pathlib import Path
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Choose "synthetic" or "sparkov".
DATASET_MODE = "sparkov"

project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root.parent != project_root:
    project_root = project_root.parent

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from fraud_lens.ingest import load_paths_config, load_sparkov_config

default_cfg = load_paths_config().get("data", {})
sparkov_cfg = load_sparkov_config().get("sparkov", {})

if DATASET_MODE == "sparkov":
    raw_path = (project_root / sparkov_cfg.get("normalized_raw_path", "data/raw_sparkov")).resolve()
    raw_glob = "*.json"
    bronze_path = (project_root / sparkov_cfg.get("bronze_path", "data/benchmark/bronze_sparkov")).resolve()
    silver_path = (project_root / sparkov_cfg.get("silver_path", "data/benchmark/silver_sparkov")).resolve()
    gold_path = (project_root / sparkov_cfg.get("gold_path", "data/benchmark/gold_sparkov")).resolve()
else:
    raw_path = (project_root / default_cfg.get("raw", "data/raw")).resolve()
    raw_glob = "*.jsonl"
    bronze_path = (project_root / default_cfg.get("bronze", "data/bronze")).resolve()
    silver_path = (project_root / default_cfg.get("silver", "data/silver")).resolve()
    gold_path = (project_root / default_cfg.get("gold", "data/gold")).resolve()

spark_builder = SparkSession.builder.appName("FraudLens-Data-Quality")
if DATASET_MODE == "sparkov":
    for key, value in sparkov_cfg.get("spark_runtime", {}).items():
        spark_builder = spark_builder.config(key, str(value))
else:
    spark_builder = spark_builder.config("spark.sql.adaptive.enabled", "true")

spark = spark_builder.getOrCreate()

print(f"Dataset mode: {DATASET_MODE}")


In [ ]:
# Load the current pipeline outputs.

raw_df = spark.read.json(str(raw_path / raw_glob))
bronze_df = spark.read.parquet(str(bronze_path))
silver_df = spark.read.parquet(str(silver_path))
gold_df = spark.read.parquet(str(gold_path))

print("Loaded layers:")
print(f"  Raw:    {raw_path}")
print(f"  Bronze: {bronze_path}")
print(f"  Silver: {silver_path}")
print(f"  Gold:   {gold_path}")


In [ ]:
# Row counts and schema checkpoints.

layer_counts = [
    ("raw", raw_df.count()),
    ("bronze", bronze_df.count()),
    ("silver", silver_df.count()),
    ("gold", gold_df.count()),
]

print("Row counts:")
for layer, count in layer_counts:
    print(f"  {layer:<6} {count:,}")

print("\nSilver schema:")
silver_df.printSchema()

print("Gold schema:")
gold_df.printSchema()


In [ ]:
# Compact label coverage summary.

coverage = gold_df.agg(
    F.count("*").alias("gold_rows"),
    F.sum(F.col("ref_transaction_id").isNotNull().cast("int")).alias("rows_with_ref_transaction_id"),
    F.sum((F.col("anomaly_type") == "impossible_travel").cast("int")).alias("impossible_travel_rows"),
    F.sum((F.col("anomaly_type") == "spending_spike").cast("int")).alias("spending_spike_rows"),
    F.sum((F.col("anomaly_type") == "fraud").cast("int")).alias("fraud_rows"),
    F.sum(F.col("speed_from_prev_kmh").isNull().cast("int")).alias("rows_with_null_speed"),
    F.sum(F.col("prior_amount_zscore").isNull().cast("int")).alias("rows_with_null_prior_amount_zscore"),
    F.sum(F.col("customer_to_merchant_distance_km").isNull().cast("int")).alias("rows_with_null_customer_distance"),
    F.sum(F.col("merchant_prior_fraud_rate").isNull().cast("int")).alias("rows_with_null_merchant_prior_fraud_rate"),
).first()

print("Coverage summary:")
for key in coverage.asDict():
    print(f"  {key}: {coverage[key]:,}")

print("\nAnomaly distribution:")
gold_df.groupBy("anomaly_type").count().orderBy("anomaly_type").show(truncate=False)

if DATASET_MODE == "sparkov":
    print("\nBenchmark label distribution:")
    gold_df.groupBy("is_fraud").count().orderBy("is_fraud").show(truncate=False)


## Rule Alignment

These checks adapt to the current dataset mode:

- synthetic mode keeps explicit impossible-travel and spending-spike checks
- Sparkov mode treats `fraud` as a generic benchmark label and avoids assuming it is a travel-specific label


In [ ]:
# Feature sanity checks.

spike_threshold = 3.0

if DATASET_MODE == "sparkov":
    feature_summary = gold_df.groupBy("is_fraud").agg(
        F.count("*").alias("rows"),
        F.expr("percentile_approx(amount_zscore, 0.5)").alias("amount_zscore_p50"),
        F.expr("percentile_approx(amount_zscore, 0.9)").alias("amount_zscore_p90"),
        F.expr("percentile_approx(prior_amount_zscore, 0.5)").alias("prior_amount_zscore_p50"),
        F.expr("percentile_approx(prior_amount_zscore, 0.9)").alias("prior_amount_zscore_p90"),
        F.expr("percentile_approx(amount_sum_last_1h, 0.5)").alias("amount_sum_last_1h_p50"),
        F.expr("percentile_approx(amount_sum_last_1h, 0.9)").alias("amount_sum_last_1h_p90"),
        F.expr("percentile_approx(tx_count_last_1h, 0.5)").alias("tx_count_last_1h_p50"),
        F.expr("percentile_approx(tx_count_last_1h, 0.9)").alias("tx_count_last_1h_p90"),
        F.expr("percentile_approx(merchant_prior_fraud_rate, 0.5)").alias("merchant_prior_fraud_rate_p50"),
        F.expr("percentile_approx(merchant_prior_fraud_rate, 0.9)").alias("merchant_prior_fraud_rate_p90"),
        F.expr("percentile_approx(customer_to_merchant_distance_km, 0.5)").alias("customer_distance_p50"),
        F.expr("percentile_approx(customer_to_merchant_distance_km, 0.9)").alias("customer_distance_p90"),
    ).orderBy("is_fraud")
    print("Benchmark feature summary by fraud label:")
    feature_summary.show(truncate=False)

    amount_quality = (
        gold_df.select(
            F.col("is_fraud"),
            (F.col("amount_zscore") >= spike_threshold).alias("amount_flag"),
            (F.col("prior_amount_zscore") >= spike_threshold).alias("prior_amount_flag"),
        )
        .groupBy("is_fraud")
        .agg(
            F.count("*").alias("rows"),
            F.sum(F.col("amount_flag").cast("int")).alias("rows_above_amount_threshold"),
            F.sum(F.col("prior_amount_flag").cast("int")).alias("rows_above_prior_amount_threshold"),
        )
        .withColumn("frac_above_amount_threshold", F.round(F.col("rows_above_amount_threshold") / F.col("rows"), 4))
        .withColumn("frac_above_prior_amount_threshold", F.round(F.col("rows_above_prior_amount_threshold") / F.col("rows"), 4))
        .orderBy("is_fraud")
    )
    print(f"Benchmark amount threshold summary (zscore >= {spike_threshold}):")
    amount_quality.show(truncate=False)

    print("Top non-fraud rows by prior_amount_zscore:")
    gold_df.where(F.col("is_fraud") == 0).orderBy(F.desc("prior_amount_zscore")).select(
        "transaction_id", "card_id", "amount", "amount_zscore", "prior_amount_zscore", "amount_sum_last_1h"
    ).show(20, truncate=False)
else:
    spike_quality = (
        gold_df.select(
            "anomaly_type",
            (F.col("amount_zscore") >= spike_threshold).alias("spike_by_rule"),
        )
        .groupBy("anomaly_type")
        .agg(
            F.count("*").alias("rows"),
            F.sum(F.col("spike_by_rule").cast("int")).alias("spike_by_rule_rows"),
        )
        .withColumn("spike_by_rule_frac", F.round(F.col("spike_by_rule_rows") / F.col("rows"), 4))
        .orderBy("anomaly_type")
    )
    print(f"Spending spike rule summary (amount_zscore >= {spike_threshold}):")
    spike_quality.show(truncate=False)


In [ ]:
# Compact distribution views.

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def clipped_series(series, lower_q=None, upper_q=0.99):
    series = pd.Series(series).dropna()
    if series.empty:
        return series
    lower = series.quantile(lower_q) if lower_q is not None else None
    upper = series.quantile(upper_q) if upper_q is not None else None
    return series.clip(lower=lower, upper=upper)

if DATASET_MODE == "sparkov":
    plot_df = (
        gold_df.select(
            F.col("is_fraud"),
            "prior_amount_zscore",
            "amount_sum_last_1h",
            "merchant_prior_fraud_rate",
            "customer_to_merchant_distance_km",
        )
        .withColumn(
            "label",
            F.when(F.col("is_fraud") == 1, F.lit("fraud")).otherwise(F.lit("non_fraud")),
        )
        .sampleBy("label", fractions={"fraud": 1.0, "non_fraud": 0.03}, seed=44)
        .toPandas()
    )

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = {"non_fraud": "#4c78a8", "fraud": "#e45756"}

    for label in ["non_fraud", "fraud"]:
        subset = plot_df[plot_df["label"] == label]
        clipped = clipped_series(subset["prior_amount_zscore"], lower_q=0.01, upper_q=0.99)
        axes[0, 0].hist(clipped, bins=50, alpha=0.55, label=label, color=colors[label])
    axes[0, 0].axvline(3.0, color="black", linestyle="--", linewidth=1)
    axes[0, 0].set_title("Prior Amount Z-Score (1st-99th pct clipped)")
    axes[0, 0].set_xlabel("prior_amount_zscore")
    axes[0, 0].legend()

    for label in ["non_fraud", "fraud"]:
        subset = plot_df[plot_df["label"] == label]
        axes[0, 1].hist(np.log1p(subset["amount_sum_last_1h"].dropna()), bins=50, alpha=0.55, label=label, color=colors[label])
    axes[0, 1].set_title("log1p(amount_sum_last_1h)")
    axes[0, 1].set_xlabel("log1p(amount_sum_last_1h)")
    axes[0, 1].legend()

    for label in ["non_fraud", "fraud"]:
        subset = plot_df[plot_df["label"] == label]
        clipped = clipped_series(subset["merchant_prior_fraud_rate"].clip(0, 1), upper_q=0.99)
        axes[1, 0].hist(clipped, bins=50, alpha=0.55, label=label, color=colors[label])
    axes[1, 0].set_title("Merchant Prior Fraud Rate (99th pct clipped)")
    axes[1, 0].set_xlabel("merchant_prior_fraud_rate (clipped to [0, 1])")
    axes[1, 0].legend()

    for label in ["non_fraud", "fraud"]:
        subset = plot_df[plot_df["label"] == label]
        clipped = clipped_series(subset["customer_to_merchant_distance_km"], upper_q=0.99)
        axes[1, 1].hist(clipped, bins=50, alpha=0.55, label=label, color=colors[label])
    axes[1, 1].set_title("Customer to Merchant Distance (99th pct clipped)")
    axes[1, 1].set_xlabel("customer_to_merchant_distance_km")
    axes[1, 1].legend()

    for ax in axes.flat:
        ax.set_ylabel("count")

    fig.tight_layout()
    plt.show()
else:
    plot_df = gold_df.select("anomaly_type", "amount_zscore", "speed_from_prev_kmh")
    travel_plot_pdf = (
        plot_df.where(F.col("speed_from_prev_kmh").isNotNull())
        .withColumn(
            "travel_label",
            F.when(F.col("anomaly_type") == "impossible_travel", F.lit("impossible_travel")).otherwise(F.lit("non_travel")),
        )
        .sampleBy("travel_label", fractions={"impossible_travel": 1.0, "non_travel": 0.03}, seed=42)
        .toPandas()
    )
    amount_plot_pdf = (
        plot_df.where(F.col("amount_zscore").isNotNull())
        .withColumn(
            "amount_label",
            F.when(F.col("anomaly_type") == "spending_spike", F.lit("spending_spike")).otherwise(F.lit("non_spike")),
        )
        .sampleBy("amount_label", fractions={"spending_spike": 1.0, "non_spike": 0.03}, seed=43)
        .toPandas()
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for label in ["non_travel", "impossible_travel"]:
        subset = travel_plot_pdf[travel_plot_pdf["travel_label"] == label]
        if not subset.empty:
            speed_values = clipped_series(np.log1p(subset["speed_from_prev_kmh"]), upper_q=0.99)
            axes[0].hist(speed_values, bins=50, alpha=0.55, label=label)
    for threshold in [250, 500, 1000]:
        axes[0].axvline(np.log1p(threshold), color="black", linestyle="--", linewidth=1)
    axes[0].set_title("Travel Speed Distribution (log1p, 99th pct clipped)")
    axes[0].set_xlabel("log1p(speed_from_prev_kmh)")
    axes[0].set_ylabel("count")
    axes[0].legend()

    for label in ["non_spike", "spending_spike"]:
        subset = amount_plot_pdf[amount_plot_pdf["amount_label"] == label]
        if not subset.empty:
            amount_values = clipped_series(subset["amount_zscore"], lower_q=0.01, upper_q=0.99)
            axes[1].hist(amount_values, bins=50, alpha=0.55, label=label)
    axes[1].axvline(3.0, color="black", linestyle="--", linewidth=1)
    axes[1].set_title("Amount Z-Score Distribution (1st-99th pct clipped)")
    axes[1].set_xlabel("amount_zscore")
    axes[1].set_ylabel("count")
    axes[1].legend()

    fig.tight_layout()
    plt.show()


In [ ]:
# Secondary diagnostics.

speed_thresholds = [250.0, 500.0, 1000.0]

if DATASET_MODE == "sparkov":
    fraud_df = gold_df.where(F.col("is_fraud") == 1)
    non_fraud_df = gold_df.where(F.col("is_fraud") == 0)

    print("Velocity summary by fraud label:")
    gold_df.groupBy("is_fraud").agg(
        F.expr("percentile_approx(tx_count_last_5m, 0.5)").alias("tx_count_last_5m_p50"),
        F.expr("percentile_approx(tx_count_last_15m, 0.5)").alias("tx_count_last_15m_p50"),
        F.expr("percentile_approx(tx_count_last_1h, 0.5)").alias("tx_count_last_1h_p50"),
        F.expr("percentile_approx(tx_count_last_1h, 0.9)").alias("tx_count_last_1h_p90"),
        F.expr("percentile_approx(card_merchant_repeat_count_prior, 0.5)").alias("card_merchant_repeat_count_prior_p50"),
    ).orderBy("is_fraud").show(truncate=False)

    print("Travel-speed summary by fraud label:")
    gold_df.groupBy("is_fraud").agg(
        F.sum(F.col("speed_from_prev_kmh").isNull().cast("int")).alias("rows_with_null_speed"),
        F.expr("percentile_approx(speed_from_prev_kmh, 0.5)").alias("median_speed"),
        F.expr("percentile_approx(speed_from_prev_kmh, 0.9)").alias("p90_speed"),
        F.max("speed_from_prev_kmh").alias("max_speed"),
    ).orderBy("is_fraud").show(truncate=False)
    threshold_rows = []
    for threshold in speed_thresholds:
        threshold_rows.append((
            threshold,
            fraud_df.where(F.col("speed_from_prev_kmh") >= threshold).count(),
            non_fraud_df.where(F.col("speed_from_prev_kmh") >= threshold).count(),
        ))
    threshold_summary = spark.createDataFrame(
        threshold_rows,
        ["speed_threshold_kmh", "fraud_rows_above_threshold", "non_fraud_rows_above_threshold"],
    )
    print("Travel-speed threshold sweep:")
    threshold_summary.show(truncate=False)
    print("Fastest non-fraud rows:")
    non_fraud_df.where(F.col("speed_from_prev_kmh").isNotNull()).orderBy(F.desc("speed_from_prev_kmh")).select(
        "transaction_id", "card_id", "event_time", "distance_from_prev_km", "hours_since_prev", "speed_from_prev_kmh"
    ).show(20, truncate=False)
else:
    travel_df = gold_df.where(F.col("anomaly_type") == "impossible_travel")
    non_travel_df = gold_df.where(F.col("anomaly_type") != "impossible_travel")
    print("Impossible-travel speed summary:")
    travel_df.agg(
        F.count("*").alias("travel_rows"),
        F.sum(F.col("ref_transaction_id").isNotNull().cast("int")).alias("travel_rows_with_ref"),
        F.sum(F.col("speed_from_prev_kmh").isNull().cast("int")).alias("travel_rows_with_null_speed"),
        F.min("speed_from_prev_kmh").alias("min_speed"),
        F.expr("percentile_approx(speed_from_prev_kmh, 0.5)").alias("median_speed"),
        F.max("speed_from_prev_kmh").alias("max_speed"),
    ).show(truncate=False)
    threshold_rows = []
    for threshold in speed_thresholds:
        threshold_rows.append((
            threshold,
            travel_df.where(F.col("speed_from_prev_kmh") < threshold).count(),
            non_travel_df.where(F.col("speed_from_prev_kmh") >= threshold).count(),
        ))
    threshold_summary = spark.createDataFrame(
        threshold_rows,
        ["speed_threshold_kmh", "labeled_travel_below_threshold", "unlabeled_rows_above_threshold"],
    )
    print("Threshold sweep:")
    threshold_summary.show(truncate=False)


In [ ]:
# Cleanup when you are done.

spark.stop()
